# Create a household BN pool

This is the supported entry point for learning the household Bayesian network in either seed-only or census-prior mode, optionally adding the unseen `Negative income` state, storing the model, and creating a condensed pool for direct-count SAA.

The graph is always learned only from states observed in the seed. In `seed` mode, observed-state CPDs also come only from the weighted seed; an optional unseen-state probability is an explicit modelling assumption. In `census` mode, parameters combine the weighted seed with a soft Dirichlet prior derived from all five census marginals.

In [1]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

import numpy as np
import pandas as pd
import pgmpy

from PopSynthesis.Methods.BN.utils.bn_pool import (
    file_sha256,
    learn_bn,
    load_bn_artifact,
    sample_condensed_pool,
    sample_zonal_exact_marginal,
    save_bn_artifact,
)

/Users/bobla/Documents/repos/PopSyn_Monash/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Configuration

Choose `BN_MODE="seed"` to learn structure and all observed-state probabilities only from the weighted seed. If Negative income is enabled in this mode, its probability is the explicit modelling assumption `SEED_ONLY_NEGATIVE_INCOME_PROBABILITY`; no census value enters model learning. Choose `BN_MODE="census"` for the existing all-attribute census Dirichlet prior.

In [2]:
REPO_ROOT = next(
    path
    for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "pyproject.toml").exists()
)

SEED_FILE = REPO_ROOT / "IO/processed_data/ori_sample_hh.csv"
CENSUS_FILE = REPO_ROOT / "IO/processed_data/census_POA_hh_saa.csv"
MODEL_FILE = REPO_ROOT / "IO/models/hh_bn/household_bn.xml"
CONDENSED_POOL_FILE = REPO_ROOT / "IO/processed_data/HH_pool_condensed.csv"
RAW_POOL_FILE = REPO_ROOT / "IO/processed_data/HH_pool.csv"
BN_POPULATION_FILE = REPO_ROOT / "IO/output/hh_bn_exact_hhsize.csv"

ATTRIBUTES = ["dwelltype", "owndwell", "hhinc", "totalvehs", "hhsize"]
BN_MODE = "seed"  # "seed" or "census"
INCLUDE_NEGATIVE_INCOME = True
SEED_ONLY_NEGATIVE_INCOME_PROBABILITY = 0.00001 # If use seed
POOL_SIZE = 2_000_000
CHUNK_SIZE = 100_000
RANDOM_SEED = 42
N_JOBS = 1
EXPORT_RAW_POOL = False
BUILD_CENSUS_MATCHED_POPULATION = True
EXACT_MARGINAL_METHOD = "auto"  # conditional root sampling, else rejection
RUN_SAA = False

## Load inputs and learn the model

In `census` mode, for each node $X$ and parent configuration $u$, the local pseudo-count is:

$$\alpha(x,u)=N_{census}P_{census}(X=x)\prod_{P\in Pa(X)}P_{census}(P=p).$$

Weighted seed counts are added to these pseudo-counts, so fitted marginals are a soft census/seed blend rather than exact census controls. In `seed` mode, observed-state CPDs use weighted maximum likelihood from the seed; an optional unseen-state probability only reserves explicit probability mass for that state.

In [3]:
seed = pd.read_csv(SEED_FILE)
census = pd.read_csv(CENSUS_FILE, header=[0, 1])

extra_states = (
    {"hhinc": ["Negative income"]}
    if INCLUDE_NEGATIVE_INCOME
    else None
)
ignored_census_states = (
    None
    if INCLUDE_NEGATIVE_INCOME
    else {"hhinc": ["Negative income"]}
)

unseen_state_probabilities = (
    {"hhinc": {"Negative income": SEED_ONLY_NEGATIVE_INCOME_PROBABILITY}}
    if BN_MODE == "seed" and INCLUDE_NEGATIVE_INCOME
    else None
)

if BN_MODE not in {"seed", "census"}:
    raise ValueError("BN_MODE must be 'seed' or 'census'")
if EXACT_MARGINAL_METHOD not in {"auto", "conditional", "root", "rejection"}:
    raise ValueError("Invalid EXACT_MARGINAL_METHOD")
if BN_MODE == "seed" and INCLUDE_NEGATIVE_INCOME:
    if not 0 < SEED_ONLY_NEGATIVE_INCOME_PROBABILITY < 1:
        raise ValueError("SEED_ONLY_NEGATIVE_INCOME_PROBABILITY must be between 0 and 1")

bn_result = learn_bn(
    seed=seed,
    attributes=ATTRIBUTES,
    mode=BN_MODE,
    census=census if BN_MODE == "census" else None,
    extra_states=extra_states,
    unseen_state_probabilities=unseen_state_probabilities,
    ignored_census_states=ignored_census_states,
    weight_column="_weight",
    prior_strength=None,  # census mode: defaults to total census households
    scoring_method="bic-d",
    n_jobs=N_JOBS,
)
model = bn_result.model

print("Edges:", sorted(model.edges()))
print("BN mode:", bn_result.mode)
if bn_result.mode == "census":
    print("Census prior strength:", f"{bn_result.prior_strength:,.0f}")
else:
    print("Census used in model learning: False")
print("Negative income supported:", "Negative income" in bn_result.state_names["hhinc"])

INFO:pgmpy: Datatype (N=numerical, C=Categorical Unordered, O=Categorical Ordered) inferred from data: 
 {'dwelltype': 'C', 'owndwell': 'C', 'hhinc': 'C', 'totalvehs': 'C', 'hhsize': 'C'}
INFO:pgmpy: Datatype (N=numerical, C=Categorical Unordered, O=Categorical Ordered) inferred from data: 
 {'dwelltype': 'C', 'owndwell': 'C', 'hhinc': 'C', 'totalvehs': 'C', 'hhsize': 'C'}
INFO:pgmpy: Datatype (N=numerical, C=Categorical Unordered, O=Categorical Ordered) inferred from data: 
 {'dwelltype': 'C', 'owndwell': 'C', 'hhinc': 'C', 'totalvehs': 'C', 'hhsize': 'C'}
  0%|          | 5/1000000 [00:00<2:07:58, 130.23it/s]
INFO:pgmpy: Datatype (N=numerical, C=Categorical Unordered, O=Categorical Ordered) inferred from data: 
 {'dwelltype': 'C', 'owndwell': 'C', 'hhinc': 'C', 'totalvehs': 'C', 'hhsize': 'C', '_weight': 'N'}


Edges: [('dwelltype', 'owndwell'), ('hhsize', 'hhinc'), ('hhsize', 'owndwell'), ('totalvehs', 'dwelltype'), ('totalvehs', 'hhsize')]
BN mode: seed
Census used in model learning: False
Negative income supported: True


In [4]:
diagnostic_rows = []
for attribute in ATTRIBUTES:
    for state in bn_result.state_names[attribute]:
        fitted_probability = bn_result.diagnostics["fitted_marginals"][attribute][state]
        row = {
            "attribute": attribute,
            "state": state,
            "fitted_probability": fitted_probability,
        }
        if bn_result.mode == "census":
            census_probability = bn_result.diagnostics["census_marginals"][attribute][state]
            row["census_probability"] = census_probability
            row["difference"] = fitted_probability - census_probability
        diagnostic_rows.append(row)

diagnostics = pd.DataFrame(diagnostic_rows)
display(diagnostics)

,attribute,state,fitted_probability
0,dwelltype,Flat or Apartment,0.142411
1,dwelltype,Missing,0.001102
2,dwelltype,Other,0.052329
3,dwelltype,Separate House,0.699487
4,dwelltype,Terrace/Townhouse,0.104670
5,owndwell,Being Purchased,0.308282
6,owndwell,Being Rented,0.256474
7,owndwell,Fully Owned,0.421018
8,owndwell,Something Else,0.014225
9,hhinc,1-149,0.012468


## Store the model and metadata

In [5]:
metadata = {
    "format_version": 1,
    "attributes": ATTRIBUTES,
    "state_names": bn_result.state_names,
    "edges": bn_result.diagnostics["edges"],
    "include_negative_income": INCLUDE_NEGATIVE_INCOME,
    "bn_mode": bn_result.mode,
    "prior_type": "dirichlet" if bn_result.mode == "census" else "seed_mle",
    "prior_strength": bn_result.prior_strength,
    "seed_only_unseen_state_probabilities": unseen_state_probabilities,
    "prior_formula": (
        "strength * P(X=x) * product(P(parent=p))"
        if bn_result.mode == "census"
        else None
    ),
    "structure_score": "bic-d",
    "weight_column": "_weight",
    "pool_size": POOL_SIZE,
    "chunk_size": CHUNK_SIZE,
    "random_seed": RANDOM_SEED,
    "model_inputs": {
        "seed_file": str(SEED_FILE.relative_to(REPO_ROOT)),
        "seed_sha256": file_sha256(SEED_FILE),
        "census_file": (
            str(CENSUS_FILE.relative_to(REPO_ROOT))
            if bn_result.mode == "census"
            else None
        ),
        "census_sha256": (
            file_sha256(CENSUS_FILE) if bn_result.mode == "census" else None
        ),
    },
    "population_controls": {
        "census_file": str(CENSUS_FILE.relative_to(REPO_ROOT)),
        "census_sha256": file_sha256(CENSUS_FILE),
    },
    "versions": {
        "pgmpy": pgmpy.__version__,
        "pandas": pd.__version__,
        "numpy": np.__version__,
    },
    "diagnostics": bn_result.diagnostics,
}

model_path, metadata_path = save_bn_artifact(model, metadata, MODEL_FILE)
print("Wrote", model_path)
print("Wrote", metadata_path)

Wrote /Users/bobla/Documents/repos/PopSyn_Monash/IO/models/hh_bn/household_bn.xml
Wrote /Users/bobla/Documents/repos/PopSyn_Monash/IO/models/hh_bn/household_bn.metadata.json


## Create the condensed pool

Sampling is chunked so a multi-million-record raw DataFrame is never retained. The output contains one row per unique household record and an integer `count` column.

In [6]:
required_states = (
    {"hhinc": ["Negative income"]}
    if INCLUDE_NEGATIVE_INCOME
    else None
)
condensed_pool = sample_condensed_pool(
    model=model,
    size=POOL_SIZE,
    chunk_size=CHUNK_SIZE,
    seed=RANDOM_SEED,
    required_states=required_states,
    attributes=ATTRIBUTES,
)

CONDENSED_POOL_FILE.parent.mkdir(parents=True, exist_ok=True)
condensed_pool.to_csv(CONDENSED_POOL_FILE, index=False)
assert int(condensed_pool["count"].sum()) == POOL_SIZE
assert not condensed_pool.duplicated(ATTRIBUTES).any()

if EXPORT_RAW_POOL:
    raw_pool = (
        condensed_pool.loc[condensed_pool.index.repeat(condensed_pool["count"]), ATTRIBUTES]
        .reset_index(drop=True)
    )
    raw_pool.to_csv(RAW_POOL_FILE, index=False)
    print("Wrote optional raw pool:", RAW_POOL_FILE)

print(f"Wrote {len(condensed_pool):,} unique records representing {POOL_SIZE:,} households")
display(condensed_pool.head())
display(
    condensed_pool.groupby("hhinc")["count"]
    .sum()
    .sort_values(ascending=False)
)

Wrote 9,455 unique records representing 2,000,000 households


,dwelltype,owndwell,hhinc,totalvehs,hhsize,count
0,Flat or Apartment,Being Purchased,1-149,0,1,128
1,Flat or Apartment,Being Purchased,1-149,0,2,33
2,Flat or Apartment,Being Purchased,1-149,0,3,11
3,Flat or Apartment,Being Purchased,1-149,0,4,1
4,Flat or Apartment,Being Purchased,1-149,0,5,1


hhinc
2500-2999          213693
1000-1249          177212
1250-1499          158938
2000-2499          158635
1750-1999          157561
500-649            142408
Nil income         137839
800-999            129403
650-799            129311
3500-3999           93283
1500-1749           89301
3000-3499           88925
300-399             70615
5000-5999           55764
150-299             49918
4000-4499           45736
400-499             36996
1-149               24844
6000-7999           20283
4500-4999           15172
8000+                4139
Negative income        24
Name: count, dtype: int64

## Optionally build a BN population matching household size in every zone

This step is separate from pool creation and runs only when `BUILD_CENSUS_MATCHED_POPULATION=True`. It samples the aggregate `hhsize` demand once and partitions the conditional draws back to every zone, then verifies every zone-state count. `EXACT_MARGINAL_METHOD="auto"` uses exact conditional/root-clamped forward sampling when `hhsize` is a root; if the learned graph gives it parents, it uses rejection sampling. Set it to `"rejection"` to force rejection sampling.

In [7]:
reloaded_model = load_bn_artifact(MODEL_FILE)

if BUILD_CENSUS_MATCHED_POPULATION:
    zone_column = next(
        column for column in census.columns if column[0] == "zone_id"
    )
    hhsize_columns = [column for column in census.columns if column[0] == "hhsize"]
    zonal_hhsize_targets = census.set_index(zone_column)[hhsize_columns].copy()
    zonal_hhsize_targets.columns = [str(column[1]) for column in hhsize_columns]
    zonal_hhsize_targets.index.name = "zone_id"
    bn_population = sample_zonal_exact_marginal(
        model=reloaded_model,
        attribute="hhsize",
        zone_targets=zonal_hhsize_targets,
        zone_field="zone_id",
        method=EXACT_MARGINAL_METHOD,
        seed=RANDOM_SEED,
    )
    BN_POPULATION_FILE.parent.mkdir(parents=True, exist_ok=True)
    bn_population.to_csv(BN_POPULATION_FILE, index=False)
    print(f"Wrote {len(bn_population):,} census-hhsize-matched households")
    print("Wrote", BN_POPULATION_FILE)
    display(bn_population.head())

Wrote 2,420,221 census-hhsize-matched households
Wrote /Users/bobla/Documents/repos/PopSyn_Monash/IO/output/hh_bn_exact_hhsize.csv


,zone_id,dwelltype,hhinc,hhsize,owndwell,totalvehs
0,3000,Other,800-999,1,Being Rented,1
1,3000,Flat or Apartment,800-999,1,Fully Owned,1
2,3000,Terrace/Townhouse,500-649,1,Fully Owned,0
3,3000,Separate House,1750-1999,1,Being Rented,3
4,3000,Flat or Apartment,1750-1999,1,Being Rented,0


## Optional direct-count SAA run

The condensed pool is passed with `pool_count_column="count"`. Keep `RUN_SAA=False` while building or inspecting the pool.

In [8]:
if RUN_SAA:
    from PopSynthesis.Methods.IPSF.run import run_saa

    run_saa(
        marg_file=CENSUS_FILE,
        pool_file=CONDENSED_POOL_FILE,
        pool_count_column="count",
        zone_field="zone_id",
        output_file=REPO_ROOT / "IO/output/hh_syn_saa_counts.csv",
        considered_atts=ATTRIBUTES,
        ordered_to_adjust_atts=[
            "hhsize",
            "totalvehs",
            "hhinc",
            "dwelltype",
            "owndwell",
        ],
        stochastic_mode="count_native_ilp",
        output_format="counts",
        direct_solver_mode="two_stage",
        zone_workers=4,
    )